# Análise de nanotoxicidade em nanopartículas por algorítmo *k*-NN

**Disciplina:** Aprendizado de máquina

**Docente:** Daniel Roberto Cassar 

**Discente e autor:** Leonardo Ramos Gomes da Silva 

## Introdução
Este caderno demonstra a implementação do algorítmo de *k*-**vizinhos mais próximos** (*k*-NN) a fim de prever a nanotoxicidade de nanopartículas, baseado em diversos atributos. O banco de dados utilizado é o *"Nanoparticle Toxicity Dataset"*, disponibilizado pela plataforma Kaggle.

## Metodologia
Para analisar os dados, utilizaremos as biblioteca pandas, matplotlib, seaborn e scikit-learn.

In [ ]:
import pandas as pd
import matplotlib as plt
import seaborn as sb
import sklearn as sk

#### Tratamento e rápida análise
O banco de dados será então introduzido e tratado. Não será necessário retirar exemplos com o método `df.dropna` pois não há exemplos com dados indisponíveis.

In [ ]:
df = pd.read_csv("nanotox_dataset.csv")

df.head()

,NPs,coresize,hydrosize,surfcharge,surfarea,Ec,Expotime,dosage,e,NOxygen,class
0,Al2O3,39.7,267.0,36.3,64.7,-1.51,24,0.001,1.61,3,nonToxic
1,Al2O3,39.7,267.0,36.3,64.7,-1.51,24,0.010,1.61,3,nonToxic
2,Al2O3,39.7,267.0,36.3,64.7,-1.51,24,0.100,1.61,3,nonToxic
3,Al2O3,39.7,267.0,36.3,64.7,-1.51,24,1.000,1.61,3,nonToxic
4,Al2O3,39.7,267.0,36.3,64.7,-1.51,24,5.000,1.61,3,nonToxic


utilizando o método `describe()` para analisar a diferença média geral entre nanopartículas tóxicas e não tóxicas

In [19]:
#Describe em partículas não tóxicas
logica = df["class"] == "nonToxic"

df.loc[logica].describe()

,coresize,hydrosize,surfcharge,surfarea,Ec,Expotime,dosage,e,NOxygen
count,405.000000,405.000000,405.000000,405.000000,405.000000,405.000000,405.000000,405.000000,405.000000
mean,47.966667,522.805185,-4.405432,61.107654,-3.997210,21.977778,31.803132,1.610321,1.656790
std,31.236046,411.580450,22.644049,60.172709,0.611865,18.077172,46.244915,0.086306,0.635761
min,7.500000,74.000000,-41.600000,7.000000,-5.170000,3.000000,0.000010,1.540000,1.000000
25%,25.000000,267.000000,-11.700000,20.000000,-4.160000,6.000000,1.000000,1.540000,1.000000
50%,35.600000,310.000000,-10.700000,40.000000,-4.160000,24.000000,10.000000,1.610000,2.000000
75%,60.000000,687.000000,-2.400000,66.000000,-3.890000,24.000000,40.000000,1.650000,2.000000
max,125.000000,1843.000000,42.800000,210.000000,-1.510000,72.000000,300.000000,1.900000,3.000000


In [20]:
#Describe em partículas tóxicas
logica = df["class"] == "nonToxic"

df.loc[~logica].describe()

,coresize,hydrosize,surfcharge,surfarea,Ec,Expotime,dosage,e,NOxygen
count,476.000000,476.000000,476.000000,476.000000,476.000000,476.000000,476.000000,476.000000,476.000000
mean,63.414916,506.103571,6.787605,25.879538,-4.035924,32.123950,46.328782,1.676450,1.012605
std,34.121298,279.919506,26.901897,21.463784,0.402980,19.534783,27.966269,0.080159,0.111680
min,7.500000,236.000000,-41.600000,7.000000,-5.170000,3.000000,10.000000,1.540000,1.000000
25%,35.600000,273.400000,-11.700000,14.500000,-3.890000,24.000000,25.000000,1.650000,1.000000
50%,52.000000,360.000000,-2.400000,20.000000,-3.890000,24.000000,50.000000,1.650000,1.000000
75%,100.000000,687.000000,29.400000,27.900000,-3.890000,48.000000,50.000000,1.650000,1.000000
max,125.000000,1093.000000,42.800000,90.000000,-3.890000,72.000000,100.000000,1.900000,2.000000


Pode-se perceber, pelas médias e medianas, que, em nanopartículas tóxicas, o tamanho do núcleo tende a ser maior e o tamanho hidrodinâmico menor - ou seja, o intervalo entre esses tamanhos diminui. Além disso, a área de superfície tende a ser menor, o que está associado com uma maior carga de superfície, visível nos dados, e há um pequeno decremento na quantidade de oxigênios.

Além disso, os atributos de tempo de exposição e dosagem tendem a ser maiores em nanopartículas tóxicas.

Por fim, a eletronegatividade não varia muito ao analisar os dados gerais.

#### Algorítmo *k*-**NN**

